# RQL highway evaluation on Colab

This notebook clones a selected Git ref, creates an isolated virtual environment, runs `eval_highway.py`, and saves per-episode CSV results plus logs to Google Drive.

`eval_highway.py` writes one CSV row after every episode and resumes from the next unfinished `episode_id` when the same output file already exists (`seed + episode_id`). Keep `RUN_NAME` fixed across reconnects so the notebook points at the same Drive CSV.

In [1]:
# Mount Google Drive. Colab will prompt you to authorize access.
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Experiment parameters (edit this cell before running).
REPO_URL = "https://github.com/thowell332/RQL-Comparison.git"  #@param {type:"string"}
REPO_REF = "main"  #@param {type:"string"}
SUPERVISOR_REPO_URL = "https://github.com/thowell332/state-wise-constrained-policy-shaping.git"  #@param {type:"string"}
SUPERVISOR_REPO_REF = "thowell332/results-galore"  #@param {type:"string"}

# Paths may be repository-relative or absolute (including paths on Drive).
MODEL_PATH = "logs/dqn_soft_residual/highway-ME-basic-AddRightReward-v0_1/best_model.zip"  #@param {type:"string"}
ENV_NAME = "highway-ME-basic-AddRightRewardALL-v0"  #@param {type:"string"}
N_EPISODES = 1000  #@param {type:"integer"}
N_STEPS = None  # Used only when N_EPISODES is None; script default is 400.
SEED = 42  #@param {type:"integer"}
MODEL_TYPE = "residual"  #@param ["residual", "dqn_me"]
SAFE_DECIDE = False  #@param {type:"boolean"}

# Keep this fixed across Colab reconnects to resume the same episodes.csv.
RUN_NAME = "highway_eval_seed42"  #@param {type:"string"}
DRIVE_RESULTS_ROOT = "/content/drive/MyDrive/aaai2027/RQL-Comparison/eval_highway"  #@param {type:"string"}
RESUME = True  #@param {type:"boolean"}

In [3]:
# Clone the requested ref and create a Colab-compatible evaluation environment.
import os
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/RQL-Comparison")
SUPERVISOR_DIR = Path("/content/scps-dependency")
VENV_DIR = Path("/content/venvs/rql-comparison")

# Headless Colab rendering for pygame / highway-env.
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["OFFSCREEN_RENDERING"] = "1"

for path in (PROJECT_DIR, SUPERVISOR_DIR, VENV_DIR):
    if path.exists():
        shutil.rmtree(path)

subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", REPO_REF], check=True)
subprocess.run(["git", "clone", SUPERVISOR_REPO_URL, str(SUPERVISOR_DIR)], check=True)
subprocess.run(["git", "-C", str(SUPERVISOR_DIR), "checkout", SUPERVISOR_REPO_REF], check=True)

if not (SUPERVISOR_DIR / "supervisor" / "__init__.py").is_file():
    raise FileNotFoundError(
        f"Selected supervisor ref has no supervisor package at {SUPERVISOR_DIR / 'supervisor'}. "
        f"Use the state-wise-constrained-policy-shaping repo (same as eval_mcts_colab)."
    )


def create_venv(venv_dir: Path) -> Path:
    """Create a Colab-friendly venv that reuses the system site packages (incl. CUDA torch)."""
    venv_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["apt-get", "install", "-y", "python3-venv", f"python{sys.version_info.major}.{sys.version_info.minor}-venv"],
        check=False,
    )
    result = subprocess.run(
        [sys.executable, "-m", "venv", "--system-site-packages", str(venv_dir)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("stdlib venv failed; falling back to virtualenv")
        print(result.stderr)
        if venv_dir.exists():
            shutil.rmtree(venv_dir)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
        subprocess.run(
            [sys.executable, "-m", "virtualenv", "--system-site-packages", str(venv_dir)],
            check=True,
        )
    return venv_dir / "bin" / "python"


VENV_PYTHON = create_venv(VENV_DIR)
subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip"], check=True)
# The repository-wide requirements pin Python-3.9-era training packages. These are
# the complete dependencies needed by the highway evaluation path on current Colab.
# Residual checkpoints were pickled under NumPy 2.x (numpy._core). Do not pin
# numpy<2 or ResidualSoftDQN.load will fail. Gym 0.23.1 still runs with a warning.
EVAL_PACKAGES = [
    "numpy>=2.0", "gym==0.23.1", "scipy", "pandas", "cloudpickle",
    "matplotlib", "tensorboard", "tensorboardX", "pygame", "tqdm",
    "rich", "pyyaml",
]
result = subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "install", *EVAL_PACKAGES],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise subprocess.CalledProcessError(result.returncode, result.args)

# Prefer the vendored highway_env / local modules over any PyPI copies.
subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "uninstall", "-y", "highway-env", "stable-baselines3"],
    check=False,
)

# Make both the RQL project and the supervisor package importable.
os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(PROJECT_DIR), str(SUPERVISOR_DIR)]
    + ([os.environ["PYTHONPATH"]] if os.environ.get("PYTHONPATH") else [])
)

# eval_highway.py imports supervisor; install editable when packaging metadata exists.
subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "install", "--no-deps", "-e", str(SUPERVISOR_DIR)],
    check=True,
)

probe = subprocess.run(
    [str(VENV_PYTHON), "-c", "from supervisor import DiscreteSupervisor; print('supervisor ok')"],
    capture_output=True,
    text=True,
    env={**os.environ, "PYTHONPATH": os.environ["PYTHONPATH"]},
)
if probe.returncode != 0:
    print(probe.stdout)
    print(probe.stderr)
    raise RuntimeError("supervisor package is not importable after install")
print(probe.stdout.strip())

entry_point = PROJECT_DIR / "eval_highway.py"
if not entry_point.is_file():
    raise FileNotFoundError(f"Selected Git ref does not contain {entry_point}")
if not (PROJECT_DIR / "basic_reward.py").is_file():
    raise FileNotFoundError(
        f"Selected Git ref is missing basic_reward.py under {PROJECT_DIR}. "
        "Push basic_reward.py (and configs/*/basic_reward.json) before running."
    )
print(f"Project: {PROJECT_DIR}")
print(f"Supervisor: {SUPERVISOR_DIR}")
print(f"Python:  {VENV_PYTHON}")

stdlib venv failed; falling back to virtualenv
Error: Command '['/content/venvs/rql-comparison/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.

supervisor ok
Project: /content/RQL-Comparison
Supervisor: /content/scps-dependency
Python:  /content/venvs/rql-comparison/bin/python


In [4]:
# Run eval_highway.py with incremental CSV output on Drive (resume-safe).
import json
import os

RUN_DIR = Path(DRIVE_RESULTS_ROOT).expanduser() / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = RUN_DIR / "episodes.csv"

def resolve_path(value):
    path = Path(value).expanduser()
    return path if path.is_absolute() else PROJECT_DIR / path

model_path = resolve_path(MODEL_PATH)
if model_path.is_symlink() and not model_path.exists():
    raise FileNotFoundError(
        f"Model path is a broken symlink: {model_path}. Commit a real zip to the selected Git ref."
    )
if not model_path.is_file() or model_path.stat().st_size == 0:
    raise FileNotFoundError(
        f"Model not found: {model_path}. Commit it to the selected Git ref or use an absolute Drive path."
    )

manifest = {
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "supervisor_repo_url": SUPERVISOR_REPO_URL,
    "supervisor_repo_ref": SUPERVISOR_REPO_REF,
    "model_path": str(model_path),
    "env_name": ENV_NAME,
    "n_episodes": N_EPISODES,
    "n_steps": N_STEPS,
    "seed": SEED,
    "model_type": MODEL_TYPE,
    "safe_decide": SAFE_DECIDE,
    "output_csv": str(OUTPUT_CSV),
    "resume": RESUME,
}
(RUN_DIR / "run_parameters.json").write_text(json.dumps(manifest, indent=2))

command = [
    str(VENV_PYTHON), str(PROJECT_DIR / "eval_highway.py"),
    "--model-path", str(model_path),
    "--env-name", ENV_NAME,
    "--seed", str(SEED),
    "--model-type", MODEL_TYPE,
    "--output", str(OUTPUT_CSV),
]
if N_EPISODES is not None:
    command += ["--n-episodes", str(N_EPISODES)]
elif N_STEPS is not None:
    command += ["--n-steps", str(N_STEPS)]
if SAFE_DECIDE:
    command.append("--safe-decide")
if not RESUME:
    command.append("--no-resume")

print(f"Output CSV: {OUTPUT_CSV}")
print("$", " ".join(command))
log_mode = "a" if RESUME and OUTPUT_CSV.exists() else "w"
run_env = {
    **os.environ,
    "PYTHONUNBUFFERED": "1",
    "PYTHONPATH": os.pathsep.join([str(PROJECT_DIR), str(SUPERVISOR_DIR)]),
}
with (RUN_DIR / "evaluation.log").open(log_mode, buffering=1) as log:
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=run_env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

print(f"Results saved to: {RUN_DIR}")

Output CSV: /content/drive/MyDrive/aaai2027/RQL-Comparison/eval_highway/highway_eval_seed42/episodes.csv
$ /content/venvs/rql-comparison/bin/python /content/RQL-Comparison/eval_highway.py --model-path /content/RQL-Comparison/logs/dqn_soft_residual/highway-ME-basic-AddRightReward-v0_1/best_model.zip --env-name highway-ME-basic-AddRightRewardALL-v0 --seed 42 --model-type residual --output /content/drive/MyDrive/aaai2027/RQL-Comparison/eval_highway/highway_eval_seed42/episodes.csv --n-episodes 1000
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
2026-07-22 13:35:04.297605: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will n

CalledProcessError: Command '['/content/venvs/rql-comparison/bin/python', '/content/RQL-Comparison/eval_highway.py', '--model-path', '/content/RQL-Comparison/logs/dqn_soft_residual/highway-ME-basic-AddRightReward-v0_1/best_model.zip', '--env-name', 'highway-ME-basic-AddRightRewardALL-v0', '--seed', '42', '--model-type', 'residual', '--output', '/content/drive/MyDrive/aaai2027/RQL-Comparison/eval_highway/highway_eval_seed42/episodes.csv', '--n-episodes', '1000']' returned non-zero exit status 1.

In [ ]:
# Verify the artifacts persisted to Drive.
for artifact in sorted(path for path in RUN_DIR.rglob("*") if path.is_file()):
    print(f"{artifact.relative_to(RUN_DIR)}  ({artifact.stat().st_size:,} bytes)")